In [2]:
import hashlib
import polars as pl

df = pl.read_parquet('../data/parquets/sold_listings_20260901.parquet').lazy()

df = df.with_columns(
    pl.col('cover_photo_url').map_elements(
        lambda x: hashlib.sha256(x.encode()).hexdigest()[:16],
                    return_dtype=pl.Utf8)
    .alias('photo_key')
)

df.collect().write_parquet('../data/parquets/sold_listings_20260901.parquet')


KeyboardInterrupt: 

In [3]:
print(df.select('photo_key').collect())

shape: (4_345_274, 1)
┌──────────────────┐
│ photo_key        │
│ ---              │
│ str              │
╞══════════════════╡
│ e5e3e873f425aee3 │
│ 7b7f0c42756c6359 │
│ 74f7d1b7bd465d51 │
│ 67a00a8734654783 │
│ ecd3000ff2758f02 │
│ …                │
│ 5e5ef885b79cc604 │
│ 99c0b120bcfd3a39 │
│ 5af4bcccbd0c42d7 │
│ af939d3150e49486 │
│ 02a4df451f4437af │
└──────────────────┘


In [7]:
urls = df.select(pl.col('cover_photo_url').unique()).collect()['cover_photo_url']
filtrd = urls.filter(~urls.str.contains('cdn.fs.grailed.com'))
print(len(urls), len(filtrd))

4184123 4171863


In [8]:
(df.select(pl.col('cover_photo_url').str.extract(r'https?://([^/]+)').alias('host'))
   .group_by('host').len().sort('len', descending=True)
   .collect())

host,len
str,u32
"""media-assets.grailed.com""",4330442
"""cdn.fs.grailed.com""",12310
"""media.grailed.com""",1700
"""son.hefa.lt""",229
"""luc.hefa.lt""",169
…,…
"""image.goat.com""",1
"""images.stockx.com""",1
"""d1qz9pzgo5wm5k.cloudfront.net""",1


In [4]:
from pathlib import Path
import polars as pl
df = pl.scan_parquet('../data/parquets/sold_listings_20260901.parquet')
ROOT = Path(r"D:\GrailedImages")
pairs = df.select(["cover_photo_url", "photo_key"]).unique().collect()
urls = pairs["cover_photo_url"]
keys = pairs["photo_key"]
existing = {p.stem for p in ROOT.rglob("*.jpg")}

missing = [
    u for u, k in zip(urls.to_list(), keys.to_list())
    if k not in existing
]

In [9]:
print(len(missing))

13258


In [14]:
from urllib.parse import urlparse

missing_hosts = pl.Series([urlparse(u).netloc for u in missing]).value_counts().sort("count", descending=True)
print(missing_hosts)

shape: (13, 2)
┌───────────────────────────────┬───────┐
│                               ┆ count │
│ ---                           ┆ ---   │
│ str                           ┆ u32   │
╞═══════════════════════════════╪═══════╡
│ cdn.fs.grailed.com            ┆ 12260 │
│ son.hefa.lt                   ┆ 229   │
│ luc.hefa.lt                   ┆ 169   │
│ media-assets.grailed.com      ┆ 167   │
│ val.hefa.lt                   ┆ 159   │
│ …                             ┆ …     │
│ cdn.filestackcontent.com      ┆ 34    │
│ i.imgur.com                   ┆ 5     │
│ d1qz9pzgo5wm5k.cloudfront.net ┆ 1     │
│ image.goat.com                ┆ 1     │
│ ensio.hefa.lt                 ┆ 1     │
└───────────────────────────────┴───────┘


In [16]:
retry_urls = [u for u in missing if "media-assets.grailed.com" in u]